# Fine-tune a YOLO football detector in Google Colab

This notebook fine-tunes pretrained YOLOv8n on the manually annotated `ball` class in `playernation_ball_training.zip`, evaluates the best checkpoint, previews validation predictions, and downloads `best.pt`. Training uses 1920-pixel input to preserve the very small footballs in the 1920?1080 source images.

Before running, choose **Runtime ? Change runtime type ? T4 GPU** (or another GPU). Upload `playernation_ball_training.zip` when prompted. The dataset has 24 training images and 7 validation images, so treat metrics as a small initial check.


In [ ]:
%pip -q install ultralytics

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, zipfile, yaml

print('Upload the manually annotated dataset archive: playernation_ball_training.zip')
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    received = ', '.join(uploaded) if uploaded else '(no files)'
    raise ValueError(
        f'Expected exactly one .zip dataset archive, but received: {received}. '
        'Select playernation_ball_training.zip. The archive must include images/ and labels/ folders.'
    )

extract_dir = Path('/content/football_dataset')
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_names[0]) as archive:
    archive.extractall(extract_dir)
train_image_dirs = list(extract_dir.rglob('images/train'))
dataset_dir = train_image_dirs[0].parents[1] if train_image_dirs else None
required = ['images/train', 'images/val', 'labels/train', 'labels/val']
if dataset_dir is None or not all((dataset_dir / part).is_dir() for part in required):
    found = [str(p.relative_to(extract_dir)) for p in extract_dir.rglob('*') if p.is_dir()][:40]
    raise RuntimeError(f'Dataset folders are missing. Found: {found}')

image_suffixes = {'.jpg', '.jpeg', '.png', '.bmp'}
counts = {}
for split in ('train', 'val'):
    image_dir = dataset_dir / 'images' / split
    label_dir = dataset_dir / 'labels' / split
    images = {p.stem for p in image_dir.iterdir() if p.suffix.lower() in image_suffixes}
    labels = {p.stem for p in label_dir.glob('*.txt')}
    if not images:
        raise ValueError(f'No images found in {image_dir}')
    if images != labels:
        raise ValueError(
            f'{split}: image/label files do not match. '
            f'Missing labels: {sorted(images-labels)[:10]}, orphan labels: {sorted(labels-images)[:10]}'
        )
    box_count = 0
    for label_path in label_dir.glob('*.txt'):
        for line_number, line in enumerate(label_path.read_text().splitlines(), start=1):
            values = line.split()
            if len(values) != 5:
                raise ValueError(f'Invalid YOLO row at {label_path}:{line_number}: {line!r}')
            cls, x, y, w, h = map(float, values)
            if cls != 0 or not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                raise ValueError(f'Invalid class/box at {label_path}:{line_number}: {line!r}')
            box_count += 1
    counts[split] = (len(images), box_count)

data_yaml = Path('/content/football_ball.yaml')
data_yaml.write_text(yaml.safe_dump({
    'path': str(dataset_dir),
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'ball'},
}, sort_keys=False))
print(f'Dataset: {dataset_dir}')
print(f"Train: {counts['train'][0]} images, {counts['train'][1]} boxes")
print(f"Validation: {counts['val'][0]} images, {counts['val'][1]} boxes")
print(data_yaml.read_text())


In [ ]:
# Visual audit: inspect these boxes before spending time on training.
# Red rectangles are the labels YOLO will learn from.
import random
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

image_dir = dataset_dir / 'images/train'
label_dir = dataset_dir / 'labels/train'
image_paths = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
if not image_paths:
    raise FileNotFoundError(f'No training images found in {image_dir}')

random.seed(42)
sample_paths = random.sample(image_paths, min(12, len(image_paths)))
fig, axes = plt.subplots((len(sample_paths) + 3) // 4, 4, figsize=(16, 4 * ((len(sample_paths) + 3) // 4)))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
for ax, image_path in zip(axes, sample_paths):
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    label_path = label_dir / f'{image_path.stem}.txt'
    if label_path.exists():
        width, height = image.size
        for line in label_path.read_text().splitlines():
            values = line.split()
            if len(values) != 5:
                raise ValueError(f'Invalid YOLO label in {label_path}: {line!r}')
            cls, x, y, box_w, box_h = map(float, values)
            if cls != 0 or not (0 <= x <= 1 and 0 <= y <= 1 and 0 < box_w <= 1 and 0 < box_h <= 1):
                raise ValueError(f'Invalid class or normalized box in {label_path}: {line!r}')
            x1, y1 = (x - box_w / 2) * width, (y - box_h / 2) * height
            x2, y2 = (x + box_w / 2) * width, (y + box_h / 2) * height
            draw.rectangle((x1, y1, x2, y2), outline='red', width=3)
    ax.imshow(image)
    ax.set_title(image_path.name[:12])
    ax.axis('off')
for ax in axes[len(sample_paths):]:
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Check that every red box tightly surrounds the football. If any box misses it, fix/re-export the labels before running the training cell.')


In [ ]:
# Fine-tune pretrained YOLOv8n. 1920px input preserves small balls from the source frames.
from ultralytics import YOLO
from pathlib import Path

IMGSZ = 1920
BATCH = 2
EPOCHS = 150
PATIENCE = 30

model = YOLO('yolov8n.pt')  # COCO-pretrained YOLOv8n; fine-tuned to the single ball class.
model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    patience=PATIENCE,
    workers=2,
    project='/content/runs',
    name='playernation-ball',
    exist_ok=True,
    pretrained=True,
    seed=42,
)
best_weights = Path('/content/runs/playernation-ball/weights/best.pt')
assert best_weights.exists(), f'Expected trained checkpoint at {best_weights}'
print(f'Best checkpoint: {best_weights}')


In [ ]:
# Evaluate the best checkpoint at the same image size used for training.
from ultralytics import YOLO
import matplotlib.pyplot as plt

best_model = YOLO(str(best_weights))
metrics = best_model.val(
    data=str(data_yaml), split='val', imgsz=IMGSZ, batch=BATCH,
    device=0, plots=True, verbose=True,
)
print('\nFINAL VALIDATION METRICS')
print(f'Precision:  {metrics.box.mp:.4f} ({metrics.box.mp * 100:.1f}%)')
print(f'Recall:     {metrics.box.mr:.4f} ({metrics.box.mr * 100:.1f}%)')
print(f'mAP@50:     {metrics.box.map50:.4f} ({metrics.box.map50 * 100:.1f}%)')
print(f'mAP@50-95:  {metrics.box.map:.4f} ({metrics.box.map * 100:.1f}%)')

# Save and display example detections. Compare these boxes with the ground truth labels.
val_images = sorted((dataset_dir / 'images/val').glob('*'))
predictions = best_model.predict(
    source=[str(p) for p in val_images], imgsz=IMGSZ, conf=0.05,
    device=0, save=True, project='/content/runs/playernation-ball',
    name='val_predictions', exist_ok=True,
)
fig, axes = plt.subplots(1, min(4, len(predictions)), figsize=(16, 4))
if not hasattr(axes, 'flat'):
    axes = [axes]
for ax, result in zip(axes, predictions):
    ax.imshow(result.plot()[..., ::-1])
    ax.set_title(Path(result.path).name)
    ax.axis('off')
plt.tight_layout()
plt.show()


## Use the trained weights in the extension

The notebook fine-tunes pretrained YOLOv8n for one class at index 0 (`ball`), matching the server's class-0 ball output. Download `best.pt` below and copy it to `playernation-ai-extension/models/football_finetuned.pt`, then restart the Python server.

This dataset has 24 training images and 7 validation images. The validation metrics and prediction preview are only an initial check; add more varied, manually labeled footage and a larger held-out split before relying on the model.


In [ ]:
files.download(str(best_weights))